# 05 - Classification with MLflow

Train Logistic Regression, Random Forest, and Gradient Boosting classifiers for cardiovascular status prediction.

In [ ]:
import os

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import pandas as pd
from pyspark.sql import functions as F
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

try:
    dbutils.widgets.text('experiment_name', '/Shared/CardioTwin_AI_Classification')
    dbutils.widgets.text('minimum_quality_score', '0.40')
    experiment_name = dbutils.widgets.get('experiment_name')
    minimum_quality_score = float(dbutils.widgets.get('minimum_quality_score'))
except Exception:
    experiment_name = 'CardioTwin_AI_Classification'
    minimum_quality_score = 0.40

features_df = spark.table('cardio_ppg_features')
quality_df = spark.table('cardio_signal_quality').select('window_id', 'signal_quality_score')

model_df = (
    features_df.join(quality_df, on='window_id', how='left')
    .fillna({'signal_quality_score': 0.0})
    .filter(F.col('signal_quality_score') >= minimum_quality_score)
)

pdf = model_df.toPandas()
feature_cols = [
    'mean_amplitude', 'std_amplitude', 'min_amplitude', 'max_amplitude', 'amplitude_range',
    'peak_count', 'heart_rate_approx', 'signal_energy', 'mean_abs_change',
    'systolic_proxy', 'diastolic_proxy', 'pulse_pressure_proxy', 'signal_quality_score'
]

X = pdf[feature_cols].fillna(0.0)
y = pdf['cardiovascular_status'].astype(int)
stratify_values = y if y.value_counts().min() >= 2 else None

try:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=stratify_values)
except ValueError:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest Classifier': RandomForestClassifier(n_estimators=150, random_state=42),
    'Gradient Boosting Classifier': GradientBoostingClassifier(random_state=42),
}

mlflow.set_experiment(experiment_name)
labels = sorted(y.unique().tolist())
leaderboard = []
trained_models = {}

for model_name, model in models.items():
    with mlflow.start_run(run_name=model_name) as run:
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        metrics = {
            'accuracy': float(accuracy_score(y_test, predictions)),
            'precision_weighted': float(precision_score(y_test, predictions, average='weighted', zero_division=0)),
            'recall_weighted': float(recall_score(y_test, predictions, average='weighted', zero_division=0)),
            'f1_weighted': float(f1_score(y_test, predictions, average='weighted', zero_division=0)),
        }

        mlflow.log_param('model_name', model_name)
        mlflow.log_param('feature_count', len(feature_cols))
        mlflow.log_param('minimum_quality_score', minimum_quality_score)
        mlflow.log_metrics(metrics)

        cm_path = f'/tmp/{model_name.lower().replace(" ", "_")}_confusion_matrix.png'
        ConfusionMatrixDisplay.from_predictions(y_test, predictions, labels=labels, cmap='Blues', values_format='d')
        plt.title(model_name)
        plt.tight_layout()
        plt.savefig(cm_path, dpi=160)
        plt.close()
        mlflow.log_artifact(cm_path)

        if hasattr(model, 'feature_importances_'):
            importance_path = f'/tmp/{model_name.lower().replace(" ", "_")}_feature_importance.png'
            importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values()
            importance.plot(kind='barh', figsize=(8, 5), color='#2b8cbe')
            plt.title(f'{model_name} Feature Importance')
            plt.tight_layout()
            plt.savefig(importance_path, dpi=160)
            plt.close()
            mlflow.log_artifact(importance_path)

        mlflow.sklearn.log_model(model, artifact_path='model')
        leaderboard.append({'run_id': run.info.run_id, 'model_name': model_name, **metrics})
        trained_models[model_name] = model

leaderboard_pdf = pd.DataFrame(leaderboard).sort_values('f1_weighted', ascending=False)
display(leaderboard_pdf)

best_model_name = leaderboard_pdf.iloc[0]['model_name']
best_model = trained_models[best_model_name]
best_model_path = '/tmp/cardio_best_classification_model'
mlflow.sklearn.save_model(best_model, best_model_path)

print(f'Best classification model: {best_model_name}')
print(f'Local MLflow model saved at: {best_model_path}')
